# Notebook 1: Baseline Speculative Decoding
**Goal:** Implement speculative decoding with a *generic* (non-tuned) TinyLlama-1.1B draft model and CodeLlama-7B as the target. Log acceptance rate, tokens/sec, TTFT, and GPU memory.

This establishes **Condition 1** — our baseline to beat with domain-tuned QLoRA.

Two implementations are compared:
- **Part A:** Custom HuggingFace loop — full token-level visibility (acceptance rate per step)
- **Part B:** vLLM pipeline — realistic throughput with batching and KV-cache management

---
**Hardware needed:** T4 (16GB) — borderline. A100 recommended for comfort.

**Runtime:** ~15 min to load models + run sample prompts.

## 0. Setup

In [ ]:
# Pin versions for stable Part A inference on A100 + CUDA 13
# vLLM installed separately in Part B to avoid replacing torch
!pip install -q "transformers==4.44.2" "accelerate>=0.26.0" datasets tqdm

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No GPU found. Runtime > Change runtime type > T4 GPU")

print(f"GPU : {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
from huggingface_hub import login
login()

TARGET_MODEL_ID = "codellama/CodeLlama-7b-hf"
DRAFT_MODEL_ID  = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

# Use draft model's tokenizer — TinyLlama vocab (32000) is valid for both models
# CodeLlama has 32016 tokens; IDs 32000-32015 crash TinyLlama's embedding layer
TOKENIZER_ID = DRAFT_MODEL_ID

TEST_PROMPTS = [
    "def fibonacci(n):\n    ",
    "def binary_search(arr, target):\n    ",
    "class Stack:\n    def __init__(self):\n        ",
    "def merge_sort(arr):\n    ",
    "def is_palindrome(s):\n    ",
]

---
## Part A — Custom HuggingFace Loop
Full token-level control: logs acceptance rate per decoding step.
Use this for detailed analysis and ablations.

In [ ]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
from transformers import AutoTokenizer, AutoModelForCausalLM

print("Loading tokenizer (TinyLlama vocab — valid for both models)...")
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_ID)

print("Loading CodeLlama-7B (target, bf16)...")
target_model = AutoModelForCausalLM.from_pretrained(
    TARGET_MODEL_ID,
    torch_dtype=torch.bfloat16,
    attn_implementation="eager"
).cuda()
target_model.eval()

print("Loading TinyLlama-1.1B (draft, bf16)...")
draft_model = AutoModelForCausalLM.from_pretrained(
    DRAFT_MODEL_ID,
    torch_dtype=torch.bfloat16,
    attn_implementation="eager"
).cuda()
draft_model.eval()
print(f"Both models loaded | VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB")

In [ ]:
import time, json
import torch.nn.functional as F
from dataclasses import dataclass, field
from typing import List

@dataclass
class RunMetrics:
    condition: str
    total_tokens: int = 0
    draft_tokens: int = 0
    accepted_tokens: int = 0
    elapsed_sec: float = 0.0
    ttft_ms: float = 0.0
    peak_gpu_gb: float = 0.0
    step_rates: List[float] = field(default_factory=list)

    @property
    def acceptance_rate(self):
        return self.accepted_tokens / self.draft_tokens if self.draft_tokens > 0 else 0.0

    @property
    def tokens_per_sec(self):
        return self.total_tokens / self.elapsed_sec if self.elapsed_sec > 0 else 0.0

    def summary(self):
        print(f"\n{'='*50}")
        print(f"Condition       : {self.condition}")
        print(f"Acceptance rate : {self.acceptance_rate:.3f} ({self.acceptance_rate*100:.1f}%)")
        print(f"Tokens/sec      : {self.tokens_per_sec:.1f}")
        print(f"TTFT            : {self.ttft_ms:.1f} ms")
        print(f"Peak GPU mem    : {self.peak_gpu_gb:.2f} GB")
        print(f"Total tokens    : {self.total_tokens}")
        print(f"{'='*50}")

In [ ]:
@torch.no_grad()
def speculative_decode(prompt, gamma=5, max_new_tokens=200,
                        temperature=1.0, condition="baseline_generic"):
    device     = next(target_model.parameters()).device
    # Draft vocab is smaller (32000) — mask target logits beyond it to prevent
    # out-of-range token IDs being fed back into the draft model's embedding layer
    DRAFT_VOCAB = draft_model.config.vocab_size   # 32000 for TinyLlama

    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
    generated = input_ids.clone()
    metrics   = RunMetrics(condition=condition)
    torch.cuda.reset_peak_memory_stats()
    t_start = time.perf_counter()
    t_first = None

    while metrics.total_tokens < max_new_tokens:
        draft_ids, draft_probs_list = [], []
        ctx = generated.clone()
        for _ in range(gamma):
            logits = draft_model(ctx).logits[:, -1, :] / temperature
            probs  = F.softmax(logits, dim=-1)
            token  = torch.multinomial(probs, 1)
            draft_ids.append(token)
            draft_probs_list.append(probs[0, token.item()].item())
            ctx = torch.cat([ctx, token], dim=-1)

        draft_seq  = torch.cat(draft_ids, dim=-1)
        full_ctx   = torch.cat([generated, draft_seq], dim=-1)

        # Mask target logits beyond draft vocab before softmax
        tgt_logits = target_model(full_ctx).logits[:, generated.shape[1]-1:-1, :] / temperature
        tgt_logits[:, :, DRAFT_VOCAB:] = float('-inf')
        tgt_probs_all = F.softmax(tgt_logits, dim=-1)

        n_accepted = 0
        for i in range(gamma):
            tok = draft_seq[0, i].item()
            p, q = tgt_probs_all[0, i, tok].item(), draft_probs_list[i]
            if torch.rand(1).item() <= min(1.0, p / (q + 1e-8)):
                generated = torch.cat([generated, draft_seq[:, i:i+1]], dim=-1)
                n_accepted += 1
                metrics.total_tokens += 1
                if t_first is None: t_first = time.perf_counter()
                if metrics.total_tokens >= max_new_tokens: break
            else:
                # Mask target logits beyond draft vocab for resample pass too
                tgt_last_logits = target_model(generated).logits[:, -1, :] / temperature
                tgt_last_logits[:, DRAFT_VOCAB:] = float('-inf')
                tgt_last  = F.softmax(tgt_last_logits, dim=-1)[0]
                corrected = F.relu(tgt_probs_all[0, i] - tgt_last)
                mass      = corrected.sum()
                if mass < 1e-6:
                    tok_new = torch.multinomial(tgt_probs_all[0, i], 1).unsqueeze(0)
                else:
                    tok_new = torch.multinomial(corrected / mass, 1).unsqueeze(0)
                generated = torch.cat([generated, tok_new], dim=-1)
                metrics.total_tokens += 1
                break

        metrics.draft_tokens    += gamma
        metrics.accepted_tokens += n_accepted
        metrics.step_rates.append(n_accepted / gamma)
        if metrics.total_tokens >= max_new_tokens: break

    metrics.elapsed_sec = time.perf_counter() - t_start
    metrics.ttft_ms     = (t_first - t_start) * 1000 if t_first else 0.0
    metrics.peak_gpu_gb = torch.cuda.max_memory_allocated() / 1e9
    output = tokenizer.decode(generated[0][input_ids.shape[1]:], skip_special_tokens=True)
    return output, metrics

In [ ]:
import numpy as np, pandas as pd

custom_metrics = []
for i, prompt in enumerate(TEST_PROMPTS):
    print(f"\nPrompt {i+1}: {prompt.strip()[:40]}")
    output, m = speculative_decode(prompt, gamma=5, max_new_tokens=150)
    custom_metrics.append(m)
    m.summary()

df_custom = pd.DataFrame([{
    "prompt":          TEST_PROMPTS[i].strip()[:35],
    "acceptance_rate": m.acceptance_rate,
    "tokens_per_sec":  m.tokens_per_sec,
    "ttft_ms":         m.ttft_ms,
    "peak_gpu_gb":     m.peak_gpu_gb,
} for i, m in enumerate(custom_metrics)])

print("\n" + df_custom.to_string(index=False))
print(f"\nMean acceptance rate : {df_custom.acceptance_rate.mean():.3f}")
print(f"Mean tokens/sec      : {df_custom.tokens_per_sec.mean():.1f}")

---
## Part B — vLLM Pipeline

vLLM handles batching, KV-cache management, and continuous batching automatically.
This gives more realistic throughput numbers than the single-sequence custom loop.

vLLM also internally tracks `SpecDecodeWorkerMetrics`:
- `draft_acceptance_rate` — fraction of draft tokens accepted
- `system_efficiency` — fraction of time not wasted on rejections
- `accepted_tokens`, `draft_tokens`, `emitted_tokens`

**Note:** Free Colab T4 may run out of memory running vLLM + the custom loop simultaneously.
If so, restart runtime, skip Part A, and run Part B fresh.

In [ ]:
# Install vLLM for Part B (separate from Part A to avoid replacing Colab's torch)
!pip install -q "vllm>=0.4.0"

# Free memory from Part A before loading vLLM engine
import gc
try:
    del target_model, draft_model
    gc.collect()
    torch.cuda.empty_cache()
    print(f"Memory freed. VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB")
except NameError:
    pass  # already freed or Part A was skipped

In [ ]:
from vllm import LLM, SamplingParams

print("Building vLLM engine with speculative decoding...")
llm = LLM(
    model=TARGET_MODEL_ID,
    speculative_config={
        "model": DRAFT_MODEL_ID,
        "num_speculative_tokens": 5,
    },
    gpu_memory_utilization=0.88,
    dtype="bfloat16",
    trust_remote_code=True,
)
print("vLLM engine ready.")

In [ ]:
sampling_params = SamplingParams(max_tokens=150, temperature=1.0, top_p=0.95)

torch.cuda.reset_peak_memory_stats()
t_start  = time.perf_counter()
outputs  = llm.generate(TEST_PROMPTS, sampling_params)
elapsed  = time.perf_counter() - t_start

completions  = [o.outputs[0].text for o in outputs]
total_tokens = sum(len(o.outputs[0].token_ids) for o in outputs)
peak_gpu_gb  = torch.cuda.max_memory_allocated() / 1e9

print(f"\nvLLM generation complete")
print(f"Total tokens    : {total_tokens}")
print(f"Elapsed         : {elapsed:.2f}s")
print(f"Tokens/sec      : {total_tokens/elapsed:.1f}")
print(f"Peak GPU mem    : {peak_gpu_gb:.2f} GB")

In [ ]:
# Extract speculative decoding metrics from vLLM internals
vllm_spec_metrics = {}
try:
    engine = llm.llm_engine
    if hasattr(engine, "stat_logger") and engine.stat_logger is not None:
        stats = engine.stat_logger.spec_decode_metrics
        if stats:
            vllm_spec_metrics = {
                "draft_acceptance_rate": stats.draft_acceptance_rate,
                "system_efficiency":     stats.system_efficiency,
                "accepted_tokens":       stats.accepted_tokens,
                "draft_tokens":          stats.draft_tokens,
            }
except Exception as e:
    print(f"Could not extract vLLM spec metrics: {e}")
    print("(This is OK — vLLM stat API varies by version)")

print("\nvLLM Speculative Decoding Metrics:")
if vllm_spec_metrics:
    for k, v in vllm_spec_metrics.items():
        print(f"  {k}: {v}")
else:
    print("  Metrics not accessible via this vLLM version.")
    print("  Throughput and latency numbers above are still valid.")

## Part C — Custom Loop vs vLLM: Throughput Comparison

In [ ]:
import matplotlib.pyplot as plt

custom_tps = df_custom.tokens_per_sec.mean() if 'df_custom' in dir() else 0
vllm_tps   = total_tokens / elapsed

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Throughput
axes[0].bar(["Custom HF Loop", "vLLM"], [custom_tps, vllm_tps],
            color=["steelblue", "coral"], width=0.4)
axes[0].set_ylabel("Tokens/sec")
axes[0].set_title("Throughput: Custom vs vLLM")
for i, v in enumerate([custom_tps, vllm_tps]):
    axes[0].text(i, v + 0.5, f"{v:.1f}", ha="center")

# Custom loop: step-level acceptance rate
if 'custom_metrics' in dir() and custom_metrics:
    axes[1].plot(custom_metrics[0].step_rates, marker="o", color="steelblue")
    axes[1].set_title("Step-level Acceptance Rate (Prompt 1, Custom Loop)")
    axes[1].set_ylabel("Acceptance Rate")
    axes[1].set_xlabel("Decoding Step")

plt.tight_layout()
plt.savefig("baseline_results.png", dpi=150)
plt.show()

In [ ]:
import os, shutil

results = {
    "condition": "baseline_generic",
    "custom_loop": {
        "mean_acceptance_rate": float(df_custom.acceptance_rate.mean()) if 'df_custom' in dir() else None,
        "mean_tokens_per_sec":  float(df_custom.tokens_per_sec.mean())  if 'df_custom' in dir() else None,
    },
    "vllm": {
        "tokens_per_sec": vllm_tps,
        "peak_gpu_gb":    peak_gpu_gb,
        **vllm_spec_metrics,
    },
}
with open("results_baseline_generic.json", "w") as f:
    json.dump(results, f, indent=2)
print("Saved results_baseline_generic.json")

# Copy to Drive if mounted
DRIVE_RESULTS_DIR = "/content/drive/MyDrive/speculative-decoding-results"
if os.path.exists("/content/drive/MyDrive"):
    os.makedirs(DRIVE_RESULTS_DIR, exist_ok=True)
    shutil.copy("results_baseline_generic.json", DRIVE_RESULTS_DIR)
    shutil.copy("baseline_results.png",          DRIVE_RESULTS_DIR)
    print(f"Copied results to Drive: {DRIVE_RESULTS_DIR}")
else:
    print("Drive not mounted — download results_baseline_generic.json and baseline_results.png manually.")

print("\nBaseline Condition 1 complete!")